# Tokenizer 分词实验 · 4 模型横向对比 (Day 4)

加载 4 个模型的分词器,对每个极端用例**横向对比**它们的分词结果。
理解:中英文差异、特殊符号、数学公式、长文本截断、特殊 token、BPE vs SentencePiece。

> 注:**Qwen2.5 与 Qwen3 共用同一个分词器**(词表都是 151936),所以本质是 3 种不同分词器:
> Qwen(byte-level BPE)、Llama(byte-level BPE)、Gemma(SentencePiece)。
>
> 本地运行:`conda activate llm_exp` → 在 `Week1/` 内启动 `jupyter lab` → 用本地 `models/`,零下载。

In [ ]:
import os
from transformers import AutoTokenizer

# 定位本地 models/ 目录(逐级向上找),纯本地、零下载
def find_models():
    d = os.getcwd()
    for _ in range(6):
        cand = os.path.join(d, "models")
        if os.path.isdir(cand):
            return cand
        d = os.path.dirname(d)
    raise FileNotFoundError("找不到 models/,请在 Week1/ 项目内启动 jupyter")

M = find_models()
# 加载 4 个模型的分词器
TOKS = {
    "Qwen2.5":  AutoTokenizer.from_pretrained(os.path.join(M, "Qwen2.5-1.5B-Instruct")),
    "Qwen3":    AutoTokenizer.from_pretrained(os.path.join(M, "Qwen3-4B")),
    "Llama3.2": AutoTokenizer.from_pretrained(os.path.join(M, "Llama-3.2-3B-Instruct")),
    "Gemma4":   AutoTokenizer.from_pretrained(os.path.join(M, "gemma-4-E2B-it")),
}
print("各模型词表大小:")
for name, t in TOKS.items():
    print(f"  {name:9s} vocab_size = {t.vocab_size}")

In [ ]:
def compare(text):
    """对一段文本,横向对比 4 个模型的分词(token 数 + 切分)。"""
    print(f"文本: {text!r}")
    for name, t in TOKS.items():
        ids  = t.encode(text, add_special_tokens=False)
        toks = t.tokenize(text)
        print(f"  {name:9s} token数={len(ids):3d}  切分={toks}")
    print("-" * 78)

# 主用 Qwen 单独看细节时用
tok = TOKS["Qwen2.5"]

## 用例 1:纯中文 —— 一个字几个 token(对比 3 种分词器)

In [ ]:
compare("今天天气真好")

## 用例 2:纯英文 —— 一个词常 1 个 token

In [ ]:
compare("The quick brown fox jumps")

## 用例 3:中英混合 —— 边界如何切分

In [ ]:
compare("我在用 Python 写 code 呢")

## 用例 4:数字与单位

In [ ]:
compare("价格是 3.14 元,重量 500g,共 1234567 件")

## 用例 5:emoji —— byte-level 会拆成多字节

In [ ]:
compare("太棒了😀🎉🚀 好开心")

## 用例 6:数学公式与符号

In [ ]:
compare("∫∑√ 求解 x²+y²=z²,其中 α+β=π")

## 用例 7:特殊 token 字面量

In [ ]:
compare("<|im_start|>system 你好 <|im_end|>")

## 用例 8:连续空格 / 制表 / 换行

In [ ]:
compare("a    b\t\tc\n\nd")

## 用例 9:代码片段

In [ ]:
compare("def is_palindrome(s): return s == s[::-1]")

## 用例 10:生僻字 / 繁体

In [ ]:
compare("龘靐齉 繁體字測試 dovahkiin")

## 用例 11:中文标点混合

In [ ]:
compare("你好,世界!(测试)【中文标点】——省略号……")

## 用例 12:长文本截断 —— max_length + truncation(以 Qwen 为例)

In [ ]:
long_text = "这是一段很长的文本。" * 100
full = tok.encode(long_text)
truncated = tok(long_text, max_length=32, truncation=True)["input_ids"]
print(f"原始 token 数     : {len(full)}")
print(f"截断到 max_length : {len(truncated)}")
print(f"截断后还原文字    : {tok.decode(truncated)!r}")

## 特殊 token 对比 —— 各模型不一样

Qwen 用 `<|im_start|>`,Gemma 用 `<start_of_turn>`,Llama 用 `<|begin_of_text|>`。
各自都是**单个不可分割的 token**,id 很大。

In [ ]:
checks = {
    "Qwen2.5":  ["<|im_start|>", "<|im_end|>", "<|endoftext|>"],
    "Llama3.2": ["<|begin_of_text|>", "<|eot_id|>"],
    "Gemma4":   ["<start_of_turn>", "<end_of_turn>"],
}
for name, specials in checks.items():
    t = TOKS[name]
    print(f"{name}:")
    for s in specials:
        print(f"   {s:20s} -> id {t.convert_tokens_to_ids(s)}")
    print()

## BPE vs SentencePiece —— 中文分词对比(重点)

- **Qwen / Llama**:byte-level BPE,中文按 UTF-8 字节合并,子词显示成乱码。
- **Gemma**:SentencePiece,中文子词直接可读。

同一句中文,token 数越少 = 越省 token = 越省钱/越快。

In [ ]:
sentence = "人工智能正在改变世界"
print(f"句子: {sentence}\n")
for name, t in TOKS.items():
    ids = t.encode(sentence, add_special_tokens=False)
    print(f"{name:9s} token数={len(ids):2d}  切分={t.tokenize(sentence)}")

## 汇总对比表 —— 多句中文的平均 token 数

In [ ]:
samples = ["人工智能正在改变世界", "今天天气真好", "我喜欢自然语言处理",
           "北京是中国的首都", "深度学习需要大量数据"]
print(f"{'模型':10s}{'词表':>10s}{'总token':>9s}{'平均/句':>9s}")
print("-" * 40)
for name, t in TOKS.items():
    counts = [len(t.encode(s, add_special_tokens=False)) for s in samples]
    total = sum(counts)
    print(f"{name:10s}{t.vocab_size:>10d}{total:>9d}{total/len(samples):>9.1f}")

## 结论(要理解的点)

1. **Qwen2.5 与 Qwen3 分词完全相同**(共用 tokenizer),所以横评里这两列数字一致。
2. **中文更费 token**:一个汉字常 1–2 个 token,英文一个词常 1 个 → 中文占更多 token,API 更贵、上下文更快满。
3. **byte-level BPE(Qwen/Llama)显示乱码**:中文子词是"字节的可打印形式",不影响还原;**SentencePiece(Gemma)显示可读中文**。
4. **词表越大越省 token**:Gemma 词表 262144 最大,同一句中文往往 token 更少;三种分词器在同句中文上的 token 数不同。
5. **特殊 token 各模型不同**:Qwen `<|im_start|>`、Gemma `<start_of_turn>`、Llama `<|begin_of_text|>`,都是整 token。
6. **空白也是 token**:空格 `Ġ`、换行 `Ċ`,能被还原。